In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , SimpleRNN , Dense,Dropout


In [2]:
word_index=imdb.get_word_index()
reverse_word_index={value:key for key,value in word_index.items()}

In [3]:
model=tf.keras.models.load_model('imdb_simple_rnn_model.h5')

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 650,371 (2.48 MB)

 Trainable params: 650,369 (2.48 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [6]:
model.get_weights()

[array([[-0.05128111, -0.04071198, -0.02978931, ...,  0.09514904,
          0.04232756,  0.02545888],
        [-0.0196391 ,  0.01514353,  0.00788873, ...,  0.00377588,
          0.06107457, -0.02420712],
        [ 0.01358581,  0.01868258,  0.01510434, ...,  0.01120943,
         -0.03607167,  0.012257  ],
        ...,
        [ 0.00583257,  0.05954714, -0.00952506, ..., -0.0391866 ,
         -0.02780079,  0.01373411],
        [-0.00013937,  0.02834963, -0.02446225, ...,  0.02967763,
          0.00049173, -0.05587399],
        [-0.00788541,  0.03338947, -0.0169049 , ..., -0.0494833 ,
          0.03800146,  0.0314524 ]], dtype=float32),
 array([[ 0.07059392, -0.1471081 , -0.10024901, ..., -0.1153338 ,
         -0.11385924, -0.11196502],
        [-0.18171987,  0.09048994,  0.0816199 , ..., -0.10243125,
         -0.14826596,  0.14583547],
        [ 0.07235246,  0.21348946, -0.08270232, ..., -0.164644  ,
         -0.19376966,  0.07236698],
        ...,
        [-0.03489758, -0.15916362,  0.2

In [5]:
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

def preprocess_text(text):
    words = text.lower().split()
    
    encoded_review = []
    for word in words:
        index = word_index.get(word, 2)
        if index < 9997:
            encoded_review.append(index + 3)
        else:
            encoded_review.append(2)  # unknown token
    
    padded_review = sequence.pad_sequences([encoded_review], maxlen=100)
    return padded_review

In [6]:
## predicting on new review
def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)
    prediction=model.predict(preprocessed_input)
    sentiment='Positive' if prediction[0][0]>0.5 else 'Negative'
    return sentiment,prediction[0][0]

In [7]:
example_review="This movie was fantastic! I really enjoyed it."
sentiment,score=predict_sentiment(example_review)
print(f"Review: {example_review}\nPredicted Sentiment: {sentiment} (Score: {score:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step
Review: This movie was fantastic! I really enjoyed it.
Predicted Sentiment: Positive (Score: 0.6631)
